In [1]:
import tensorrt as trt
from tensorrt_model import TRTModel
from ssd_tensorrt import load_plugins, parse_boxes,TRT_INPUT_NAME, TRT_OUTPUT_NAME
import ctypes

from RobotClass import Robot

import numpy as np
import cv2

import os
import uuid
import time
from typing import List

import ipywidgets.widgets as widgets
from IPython.display import display, HTML

from scipy.optimize import linear_sum_assignment

In [2]:
def bgr8_to_jpeg(value):#convert numpy array to jpeg coded data for displaying 
    return bytes(cv2.imencode('.jpg',value)[1])

In [3]:
mean = 255.0 * np.array([0.5, 0.5, 0.5])
stdev = 255.0 * np.array([0.5, 0.5, 0.5])

def bgr8_to_ssd_input(camera_value):
    x = camera_value
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    x = x.transpose((2, 0, 1)).astype(np.float32)
    x -= mean[:, None, None]
    x /= stdev[:, None, None]
    return x[None, ...]

class ObjectDetector(object):
    
    PEOPLE = 1
    
    def __init__(self, engine_path, preprocess_fn=bgr8_to_ssd_input):
        logger = trt.Logger()
        trt.init_libnvinfer_plugins(logger, '')
        load_plugins()
        self.trt_model = TRTModel(engine_path, input_names=[TRT_INPUT_NAME],output_names=[TRT_OUTPUT_NAME, TRT_OUTPUT_NAME + '_1'])
        self.preprocess_fn = preprocess_fn
        
    def execute(self, *inputs):
        trt_outputs = self.trt_model(self.preprocess_fn(*inputs))
        return parse_boxes(trt_outputs)
    
    def __call__(self, *inputs):
        return self.execute(*inputs)

model = ObjectDetector('Robotics_tuto/ssd_mobilenet_v2_coco.engine')

In [4]:
def draw_debug_rect(img, rect, color=(255, 0, 0)):
    cv2.rectangle(img, (int(rect[0]), int((rect[1]))), (int(rect[2]), int(rect[3])), color, 2)

def draw_debug_rects(img, rects, color=(255, 0, 0)):
    for rect in rects:
        if len(rect) >= 5:
            draw_debug_rect(img, rect[0:4], rect[4])
        else:
            draw_debug_rect(img, rect[0:4], color)

def draw_debug_circle(img, rect, color=(255, 0, 0)):
    if len(rect) == 4:
        centerx = int((rect[0] + rect[2]) // 2)
        centery = int((rect[1] + rect[3]) // 2)
    
        cv2.circle(img, (centerx, centery), radius=3, color=color, thickness=-1)
    else:
        cv2.circle(img, (rect[0], rect[1]), radius=3, color=color, thickness=-1)
    
def draw_debug_circles(img, rects, color=(255, 0, 0)):
    for rect in rects:
        if len(rect) == 5:
            draw_debug_circle(img, rect[0:4], rect[4])
        else:
            draw_debug_circle(img, rect[0:4], color)

def draw_debug_text(img, center, text, color = (255, 0, 0)):
    cv2.putText(img, text, center, cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)

def draw_debug_track(img, track, color = (255,0,0)):
    
    color = (255,0,0)
    if track.strikes > 0:
        color = (0,0,255)
    
    draw_debug_circle(img, track.get_pos())
    bbox = (
        track.get_pos()[0] - track.get_size()[0] // 2,
        track.get_pos()[1] - track.get_size()[1] // 2,
        track.get_pos()[0] + track.get_size()[0] // 2,
        track.get_pos()[1] + track.get_size()[1] // 2,
        color
    )
    
    draw_debug_circle(img, track.get_pos(),color)
    draw_debug_text(img, track.get_pos(), str(track.id)[:4], color)

In [5]:
# https://en.wikipedia.org/wiki/Hungarian_algorithm#Implementation_in_C++
def hungarian_algorithm(
    cost_matrix: List[List[int]]
) -> tuple:
    """
        Solves the assignment problem for assigning observations
        to tracks
        
        Arguments:
            cost_matrix: A matrix of distances from a specific
                observation to a specific track
        
        Returns:
            In the first list, the ith element is the minimum cost 
                of assigning i jobs
            In the second list, the ith element is the index of the
                observation being assigned to the ith track
    """
    
#     rows, cols = linear_sum_assignment(cost_matrix)
#     rows = [x for x in rows]
#     cols = [y for y in cols]
#     pairs = list(zip(rows, cols))
#     retval = [-1] * len(pairs)
    
#     for p in pairs:
#         retval[p[0]] = p[1]

#     return (rows, cols)
    # A large number
    INFINITY = float('inf')
    
    jobs_count = len(cost_matrix)
    worker_count = len(cost_matrix[0])
    assert(worker_count >= jobs_count)
    
    job = [-1] * (worker_count+1)
    ys = [0] * jobs_count
    yt = [0] * (worker_count + 1)
    
    answers = []
    
    for job_current in range(jobs_count):
        worker_current = worker_count
        job[worker_current]= job_current
        
        min_to = [INFINITY] * (worker_count + 1)
        previous_worker = [-1] * (worker_count + 1)
        is_worker_in_z = [False] * (worker_count + 1)
        while job[worker_current] != -1:
            is_worker_in_z[worker_current] = True
            j = job[worker_current]
            delta = INFINITY
            next_worker = None
            for w in range(worker_count):
                if not is_worker_in_z[w]:
                    if min_to[w] > cost_matrix[j][w] - ys[j] - yt[w]:
                        min_to[w] = cost_matrix[j][w] - ys[j] - yt[w]
                        previous_worker[w] = worker_current
                    if delta > min_to[w]:
                        delta = min_to[w]
                        next_worker = w
            
            for w in range(worker_count+1):
                if is_worker_in_z[w]:
                    ys[job[w]] += delta
                    yt[w] -= delta
                else:
                    min_to[w] -= delta
            
            worker_current = next_worker
    
        while worker_current != worker_count:
            w = previous_worker[worker_current]
            job[worker_current] = job[w]
            worker_current = w
        answers.append(-yt[worker_count])
    return (answers, job)

In [6]:
WIDTH = 640
HEIGHT = 480

def squared_distance(p1: tuple, p2: tuple):
    return (p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2

def create_cost_matrix(
    predicted_positions:List[tuple],
    actual_positions: List[tuple]
):
    cost = [
        [ squared_distance(predicted_pos, actual_position)
         for j, actual_position
         in enumerate(actual_positions)
        ]
        for i, predicted_pos
        in enumerate(predicted_positions)
    ]
    
    return cost

def get_bbox_size(bbox):
    sizex = bbox[2] - bbox[0]
    sizey = bbox[3] - bbox[1]
    return (sizex * WIDTH, sizey * HEIGHT)

def get_bbox_center(bbox):
    centerx = (bbox[0] + bbox[2]) / 2
    centery = (bbox[1] + bbox[3]) / 2
    return (int(centerx * WIDTH), int(centery * HEIGHT))

In [31]:
MAX_MOVE_DISTANCE_ALLOWED = 100
MAX_FRAMES_UNASSIGNED = 20

class Track:
    """
        Class that holds objects being tracked by the camera.
        
        All coordinates are screen coordinates.
        
        Attributes:
            id: uuid identifing the tracks
            curr_pos: the current position of the track
            prev_pos: a history of positions of the track
            strikes: the number of times the track does not
                have a corresponding object. Used for marking
                it for deletion
    """
    
    def __init__(self, position: tuple, size: tuple = (0,0)):
        """
            Initialises track.
            
            Args:
                position: current position of the robot
        """
        self.size = size
        self.id = uuid.uuid4()
        self.curr_pos = position
        self.prev_pos = []
        self.strikes = 0
    
    def should_delete(self):
        self.strikes += 1
        if self.strikes > MAX_FRAMES_UNASSIGNED:
            return True
        return False
    
    def get_pos(self) -> tuple:
        """
            Returns current position
            
            Returns:
                current position
        """
        return self.curr_pos
    
    def set_pos(self, position: tuple) -> bool:
        """
            Attempts to set the current position of the robot,
            remembering where it was previously.
            
            It refuses and returns False if the distance is too large
            
            Args:
                position: the new position of the robot
            Returns:
                None
        """
        if squared_distance(self.curr_pos, position) > MAX_MOVE_DISTANCE_ALLOWED**2:
            return False
        self.prev_pos.append(self.curr_pos)
        if len(self.prev_pos) > 200:
            self.prev_pos = self.prev_pos[(len(self.prev_pos) - 200):]
        self.curr_pos = position
        self.strikes = 0
        return True
    
    def get_size(self) -> tuple:
        """
            Gets the width and height of the track
            
            Returns:
                Width and height of the track in a single tuple
        """
        return self.size
    
    def set_size(self, size: tuple) -> bool:
        """
            Sets the size of the tuple
            
            Args:
                size: A tuple with the width and height of the track
        """
        self.size = size
    
    
    def predict_pos(self) -> tuple:
        """
            Tries to predict where the object is going to be using
            polynomial curve fitting on the 2 axis.
            
            If there are fewer observations than the degree of the curve,
            the prediction might not be accurate so it returns the last
            known position
            
            Returns:
                the coordinates of where the object is predicted to be.
        """
        POLYNOMIAL_DEGREE = 4
        
        positions = self.prev_pos + [self.curr_pos]
        
        if len(positions) < POLYNOMIAL_DEGREE * 2 + 2:
            return positions[-1]
        
        history_id = list(range(len(positions)))
        x_list = [p[0] for p in positions]
        y_list = [p[1] for p in positions]
    
        coeffx = np.polyfit(history_id, x_list, POLYNOMIAL_DEGREE)
        coeffy = np.polyfit(history_id, y_list, POLYNOMIAL_DEGREE)
        
        predictor_x = np.poly1d(coeffx)
        predictor_y = np.poly1d(coeffy)
        
        predicted_x = predictor_x(len(positions))
        predicted_y = predictor_y(len(positions))
        
        return (predicted_x, predicted_y)

In [8]:
# Set up widgets for debugging

image_widget = widgets.Image(format='jpeg', width=300, height=300)
depth_widget = widgets.Image(format='jpeg', width=300, height=300)
label_widget = widgets.IntText(value=1, description='tracked label')
distance_label = widgets.Label("", description="Distance")
target_id_label = widgets.Label("", description="Target")
previous_target_id_label = widgets.Label("", description="Previous Target")

target_selection = widgets.Dropdown(
    options=[("None", None)],
    description='Target:',
    disabled=False,
)

display(widgets.VBox([
    widgets.HBox([image_widget,depth_widget]),
    target_selection,
    label_widget,
    distance_label,
    target_id_label,
    previous_target_id_label
]))

None None
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7f0017ccc0>
80375d13-4f63-4459-89d2-43ac552a0317 <__main__.Track object at 0x7

7b58f550-4d92-49f7-b6d0-95592fc32fbd <__main__.Track object at 0x7ed8b84828>
7b58f550-4d92-49f7-b6d0-95592fc32fbd <__main__.Track object at 0x7ed8b84828>
7b58f550-4d92-49f7-b6d0-95592fc32fbd None
Finding new target...
6eb2a284-6ba2-4a31-aca2-5277ff784252
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track object at 0x7ed8b7bbe0>
6eb2a284-6ba2-4a31-aca2-5277ff784252 <__main__.Track

In [9]:
tracks = []

def process_frame(image):
    global tracks
    
    img_resized = cv2.resize(image, (300,300))
    detections = model(img_resized)
    detections = [d for d in detections[0] if d['label'] == ObjectDetector.PEOPLE]
    detections = [d for d in detections if d['confidence'] >= 0.5]
    
    for i in detections:
        rect = [i['bbox'][0]*WIDTH, i['bbox'][1]*HEIGHT, i['bbox'][2]*WIDTH, i['bbox'][3]*HEIGHT]
        draw_debug_rect(image, rect)
    
    if len(detections) == 0:
        marked_for_deletion = set()
        for i, track in enumerate(tracks):
            if track.should_delete():
                marked_for_deletion.add(i)
        
        tracks = [track for i, track in enumerate(tracks) if not i in marked_for_deletion]
    elif len(tracks) == 0:
        for detection in detections:
            tracks.append(Track(get_bbox_center(detection['bbox'])))
    elif len(tracks) <= len(detections):
        cost_matrix = create_cost_matrix(
            [t.predict_pos() for t in tracks],
            [get_bbox_center(d['bbox']) for d in detections]
        )
        
        marked_for_deletion = set()
        
        assignments = hungarian_algorithm(cost_matrix)[1]
        for i, track in enumerate(tracks):
            detection = detections[assignments[i]]
            track.set_pos(get_bbox_center(detection['bbox']))
            track.set_size(get_bbox_size(detection['bbox']))
        
        # Find any detection that does not have a track assigned
        assignments = set(assignments)
        detections_indices = set(range(len(detections)))
        unassigned = detections_indices.difference(assignments)
        
        #Create a new track from unassigned detections
        for unassigned_index in unassigned:
            tracks.append(Track(
                get_bbox_center(detections[unassigned_index]['bbox']),
                get_bbox_size(detections[unassigned_index]['bbox'])
            ))
        
        # Delete any track that has become invalid
        tracks = [t for i, t in enumerate(tracks) if i not in marked_for_deletion]
    else:
        cost_matrix = create_cost_matrix(
            [get_bbox_center(d['bbox']) for d in detections],
            [t.predict_pos() for t in tracks]
        )
        
        assignments = hungarian_algorithm(cost_matrix)[1]
        marked_for_deletion = set()
        
        for i, detection in enumerate(detections):
            track = tracks[assignments[i]]
            if not track.set_pos(get_bbox_center(detection['bbox'])):
                if track.should_delete():
                    assignments[i] = -1
            else:
                track.set_size(get_bbox_size(detection['bbox']))
        assignments = set(assignments)
        tracks_indices = set(range(len(tracks)))
        unassigned = tracks_indices.difference(assignments)
        for trackid in unassigned:
            if tracks[trackid].should_delete():
                marked_for_deletion.add(trackid)
        tracks = [track for i, track in enumerate(tracks) if i not in marked_for_deletion]

In [10]:
PATH_TO_IMAGE_SAMPLES = "imgs"

def image_generator(path: str = PATH_TO_IMAGE_SAMPLES, start_from: int = 0):
    img_list = os.listdir(path)
    ids = [int(p[:p.find("_")]) for p in img_list]
    zipped_imgs = list(zip(ids, img_list))
    zipped_imgs.sort()
    for index, img_name in zipped_imgs:
        if index >= start_from:
            yield cv2.imread(f"{path}/{img_name}", cv2.IMREAD_COLOR)
        

In [21]:
DEPTH_MINIMUM = 1000
DEPTH_BACK = 600
OBSTACLE_DEPTH_MINIMUM = 400

TURN_POS = 50

TURN_SPEED = 0.4
MOVE_SPEED = 0.4

depth_slice = (5,40)
depth_top = 400
depth_margins = (WIDTH //2 - 100, WIDTH //2 + 100)

target_id = None
previous_target = None

def on_target_change(change):
    global target_id
    if change['new'] != None:
        target_id = change['new']

target_selection.observe(on_target_change, names=['value'])

In [12]:
DISTANCE_RETARGET = 40

def get_target(target_id):
    for track in tracks:
        if track.id == target_id:
            return track
    return None

def get_target_depth(track, depth_frame):
    bbox = (
        int(track.get_pos()[0] - track.get_size()[0] // 2),
        int(track.get_pos()[1] - track.get_size()[1] // 2),
        int(track.get_pos()[0] + track.get_size()[0] // 2),
        int(track.get_pos()[1] + track.get_size()[1] // 2)
    )
    
    depth_frame_slice = depth_frame[bbox[0]:bbox[2], bbox[1]:bbox[3]]
    depth_frame_slice[depth_frame_slice<100]=0
    depth_frame_slice[depth_frame_slice>1000]=0
    depth_frame_slice = depth_frame_slice[depth_frame_slice != 0]
    depth_min = 0
    if len(depth_frame_slice) > 0:
        depth_min = depth_frame_slice.min()
    
    return depth_min

def find_new_target(previous_target):
    new_target_id = None
    new_target = None
    min_distance = 999999
    for track in tracks:
        sdistance = squared_distance(track.get_pos(), previous_target.get_pos())
        if sdistance < min_distance:
            new_target_id = track.id
            new_target = track
    
    return new_target_id, new_target
        


In [25]:
def processing(change):
    global target_id
    global previous_target
    image = np.copy(change['new'])
    depth_frame = camera.depth_frame

    process_frame(image)

    target_selection.options = [("None", None)] + [(str(t.id)[:4], t.id) for t in tracks]
    target_track = get_target(target_id)
    if target_track == None:
        if target_id != None and previous_target != None:
            print("Finding new target...")
            target_id, target_track = find_new_target(previous_target)
            print(target_id)
            if target_track != None:
                previous_target = target_track
        else:
            target_id = None
    else:
        previous_target = target_track
    
    if previous_target:
        previous_target_id_label.value = "Prev: " + str(previous_target.id)
    
    if target_id:
        target_id_label.value = "ID: " + str(target_id)
        
    # Track debug UI
    for track in tracks:
        color = (255, 0, 0)
        if target_track and track.id == target_track.id:
            color = (0, 0, 255)
        draw_debug_track(image, track, color)


    #collision avoidance
    left_obstacle, right_obstacle = process_depth(depth_frame)
    
    if target_track:

        distance_to_target = get_target_depth(target_track, depth_frame)
        distance_label.value = "Distance: " + str(distance_to_target)

        # Collision UI
        draw_debug_rect(image, (depth_margins[0], depth_top, depth_margins[1], depth_top + depth_slice[1]))

        if left_obstacle:
            draw_debug_circle(image, (left_obstacle, depth_top), (0,255,0))
        if right_obstacle:
            draw_debug_circle(image, (right_obstacle, depth_top), (0,255,0))

        if not left_obstacle and not right_obstacle:
            if target_track.get_pos()[0] < WIDTH // 2 - TURN_POS and target_track.strikes == 0:
                robot.left(TURN_SPEED)
            elif target_track.get_pos()[0] > WIDTH // 2 + TURN_POS and target_track.strikes == 0:
                robot.right(TURN_SPEED)
            elif distance_to_target > DEPTH_MINIMUM or distance_to_target == 0:
                robot.forward(MOVE_SPEED)
            elif distance_to_target < DEPTH_BACK and distance_to_target != 0:
                robot.backward(MOVE_SPEED)
            else:
                robot.stop()
        elif left_obstacle and not right_obstacle:
            if left_obstacle > target_track.get_pos()[0]:
                robot.right(TURN_SPEED)
        elif not left_obstacle and right_obstacle:
            if right_obstacle < target_track.get_pos()[0]:
                robot.left(TURN_SPEED)
        elif target_track.get_pos()[0] - left_obstacle > right_obstacle - target_track.get_pos()[0] and right_obstacle > WIDTH // 2:
            robot.right(TURN_SPEED)
        elif target_track.get_pos()[0] - left_obstacle < right_obstacle - target_track.get_pos()[0] and left_obstacle < WIDTH // 2:
            robot.left(TURN_SPEED)
    else:
        robot.stop()
    image_widget.value = bgr8_to_jpeg(image)
        

In [14]:
tracks = []
# for image in image_generator():
#     processing({'new': image})

In [15]:
def process_depth(depth_frame):
    
    left = depth_margins[0]
    right = depth_margins[1]
    
    left_obstacle = None
    right_obstacle = None
    
    while (left < right) and (left_obstacle is None and right_obstacle is None):
        if left_obstacle is None:
            depth_frame_slice = depth_frame[
                left:left + depth_slice[0],
                depth_top:depth_top+depth_slice[1]
            ]
            depth_frame_slice[depth_frame_slice<100]=0
            depth_frame_slice[depth_frame_slice>1000]=0
            depth_frame_slice = depth_frame_slice[depth_frame_slice != 0]
            depth_min = 0
            if len(depth_frame_slice) > 0:
                depth_min = depth_frame_slice.min()
            
            if depth_min != 0 and depth_min < OBSTACLE_DEPTH_MINIMUM:
                left_obstacle = left
            else:
                left += 1
        if right_obstacle is None:
            depth_frame_slice = depth_frame[
                right - depth_slice[0]:right,
                depth_top:depth_top+depth_slice[1]
            ]
            depth_frame_slice[depth_frame_slice<100]=0
            depth_frame_slice[depth_frame_slice>1000]=0
            depth_frame_slice = depth_frame_slice[depth_frame_slice != 0]
            depth_min = 0
            if len(depth_frame_slice) > 0:
                depth_min = depth_frame_slice.min()
            
            if depth_min != 0 and depth_min < OBSTACLE_DEPTH_MINIMUM:
                right_obstacle = left
            else:
                right -= 1
    return (left_obstacle, right_obstacle)

In [16]:
import requests
import io
import os

def send_image(image, path):
    pass
#     with open(f"{path}/{len(os.listdir(path))}", "wb") as f:
#         f.write(image)
    

In [17]:
#use traitlets and widgets to display the image in Jupyter Notebook
import traitlets
from traitlets.config.configurable import SingletonConfigurable

#use opencv to covert the depth image to RGB image for displaying purpose
import cv2
import numpy as np

#using realsense to capture the color and depth image
import pyrealsense2 as rs

#multi-threading is used to capture the image in real time performance
import threading

class Camera(SingletonConfigurable):
    
    #this changing of this value will be captured by traitlets
    color_value = traitlets.Any()
    
    def __init__(self):
        super(Camera, self).__init__()
        
        #configure the color and depth sensor
        self.pipeline = rs.pipeline()
        self.configuration = rs.config()  
        
        #set resolution for the color camera
        self.color_width = 640
        self.color_height = 480
        self.color_fps = 30
        self.configuration.enable_stream(rs.stream.color, self.color_width, self.color_height, rs.format.bgr8, self.color_fps)

        #set resolution for the depth camera
        self.depth_width = 640
        self.depth_height = 480
        self.depth_fps = 30
        self.configuration.enable_stream(rs.stream.depth, self.depth_width, self.depth_height, rs.format.z16, self.depth_fps)

        #flag to control the thread
        self.thread_runnning_flag = False
        
        #start the RGBD sensor
        self.pipeline.start(self.configuration)
        self.pipeline_started = True
        frames = self.pipeline.wait_for_frames()

        #start capture the first color image
        color_frame = frames.get_color_frame()   
        image = np.asanyarray(color_frame.get_data())
        self.color_value = image

        #start capture the first depth image
        depth_frame = frames.get_depth_frame()           
        depth_image = np.asanyarray(depth_frame.get_data())
        depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET)
        self.depth_value = depth_colormap
        self.depth_frame = depth_image

    def _capture_frames(self):
        while(self.thread_runnning_flag==True): #continue until the thread_runnning_flag is set to be False
            frames = self.pipeline.wait_for_frames() #receive data from RGBD sensor
            
            color_frame = frames.get_color_frame() #get the color image
            image = np.asanyarray(color_frame.get_data()) #convert color image to numpy array
            self.color_value = image #assign the numpy array image to the color_value variable 

            depth_frame = frames.get_depth_frame() #get the depth image           
            depth_image = np.asanyarray(depth_frame.get_data()) #convert depth data to numpy array
            #conver depth data to BGR image for displaying purpose
            depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET)
            self.depth_vimage = depth_colormap #assign the color BGR image to the depth value
            self.depth_frame = depth_image
            
            send_image(bgr8_to_jpeg(image), "imgs/color")
            send_image(depth_image.tobytes(), "imgs/depth")
    
    def start(self): #start the data capture thread
        if self.thread_runnning_flag == False: #only process if no thread is running yet
            self.thread_runnning_flag=True #flag to control the operation of the _capture_frames function
            self.thread = threading.Thread(target=self._capture_frames) #link thread with the function
            self.thread.start() #start the thread

    def stop(self): #stop the data capture thread
        if self.thread_runnning_flag == True:
            self.thread_runnning_flag = False #exit the while loop in the _capture_frames
            self.thread.join() #wait the exiting of the thread       

#create a camera object
camera = Camera.instance()
robot = Robot()

In [30]:
camera.start()
camera.observe(processing, names='color_value')

In [29]:
camera.unobserve_all()
camera.stop()
tracks = []
previous_track = None
target_id = None

In [20]:
robot.stop()
tracked_id = None
tracks = []

In [1]:
from RobotClass import Robot
import time
import random

In [ ]:
robot = Robot()

In [32]:
def thing(t):
    a = 0.3 * t
    b = -1 * t
    robot.set_motors(a,a,b,b)

def thing2():
    ITER_START = 300
    ITER_END = 310
    for i in range(ITER_START, ITER_END):
        t = i / ITER_END
        thing(t**3)

def thing3():
    for i in range(6):
        thing2()
        robot.stop()
        time.sleep(random.random() * 0.3)
        
thing3()        
robot.stop()
    

In [21]:
robot.stop()

In [26]:
import random
import time

In [ ]:
class TestingRobot:
    
    IDLE = 0
    FORWARD = 1
    BACKWARD = 2
    LEFT = 4
    RIGHT = 8
    
    def __init__(self):
        self.state = IDLE
    
    def forward(self, speed = 1):
        self.state = FORWARD
    
    def backward(self, speed = 1):
        self.state = BACKWARD
    
    def left(self, speed = 1):
        self.state = LEFT
    
    def right(self, speed = 1):
        self.state = RIGHT
    
    def stop(self):
        self.state = IDLE

robot = TestingRobot()
tracks = []
target_id = None
previous_target = None
        
def test_hungarian():
    costs = [
        [8, 5, 9],
        [4, 2, 4],
        [7, 3, 8]
    ]
    
    expected = [5, 9, 15]
    
    return hungarian_algorithm(costs)[0] == expected

def test_squared_distance():
    a = (3,2)
    b = (5,0)
    
    return squared_distance(a,b) == 8

def test_create_cost_matrix():
    track_pos = [(1,1), (5,6), (5,1)]
    object_pos = [(1,2), (5,6), (3,1)]
    
    cost_matrix = create_cost_matrix(track_pos, object_pos)
    if cost_matrix[0] == [1, 41, 4] and \
        cost_matrix[1] == [1, 32, 29] and \
        cost_matrix[2] == [17, 25, 4]:
        
        return True
    
    return False

def test_get_bbox_size():
    bbox = [0.1,0.1,0.4,0.5]
    return get_bbox_size(bbox) == ( int(0.3*WIDTH), int(0.4*HEIGHT))

def test_get_bbox_center():
    bbox = [0.1,0.1,0.4,0.5]
    return get_bbox_center(bbox) == ( int(0.25*WIDTH), int(0.3*HEIGHT))

def test_get_target():
    
    results = [False, False]
    
    tracks = [Track((1,1)), Track((2,2))]
    target_id = tracks[0].id
    
    results[0] = get_target(target_id) == tracks[0]
    target_id = uuid.uuid4()
    results[1] = get_target(target_id) == None
    
    return results

def test_cnn():
    testcases = [{'img_path': '', 'people': []}]
    passed = [False] * len(testcases)
    
    for index, testcase in enumerate(testcases):
        img = cv2.imread(testcase['img_path'])
        img_resized = cv2.resize(image, (300,300))
        detections = model(img_resized)
        detections = [d for d in detections[0] if d['label'] == ObjectDetector.PEOPLE]
        predicted_centers = [get_bbox_center(d['bbox']) for d in detections]
        
        is_close_enough = [
            get_squared_distance(true_center, predicted_center) < 400
            for true_center, predicted_center in zip(testcase['people'], predicted_centers)
        ]
        
        
        
        passed[index] = all(is_close_enough)
    
    return passed


# Need images
def test_follow_human():
    testcases = [{'color_img_path': '', 'depth_img_path':'', 'expected_state': 0}]
    results = [False] * len(testcases)
    for index, testcase in enumerate(testcases):
        img = cv2.imread(testcase['color_img_path'])
        depth_img = cv2.imread(testcase['depth_img_path'])
        process(img, depth_img)
        
        results[index] = robot.state == testcase['expected_state']
    
    return results

def test_avoid_obstacle():
    testcases = [{'color_img_path': '', 'depth_img_path':'', 'expected_state': 0}]
    results = [False] * len(testcases)
    for index, testcase in enumerate(testcases):
        img = cv2.imread(testcase['color_img_path'])
        depth_img = cv2.imread(testcase['depth_img_path'])
        process(img, depth_img)
        
        results[index] = robot.state == testcase['expected_state']
    
    return results

def run_tests():
    test_functions = [
        test_hungarian,
        test_squared_distance,
        test_create_cost_matrix,
        test_get_bbox_size,
        test_get_bbox_center,
        test_get_target,
        test_get_target_depth,
        test_find_new_target,
        test_cnn,
        test_follow_human,
        test_avoid_obstacle,
    ]

    for test in test_functions:
        if test():
            print(f"Test {test.__name__} was SUCCESSFUL")
        else:
            print(f"Test {test.__name__} FAILED")